# Discover MerLin by Reproducing a QRC Paper

## Interactive workshop — 2 hours

**Case study:** *Level Generation with Quantum Reservoir Computing*  
**Goal:** generate a Super Mario Bros. level column by column using a **MerLin photonic quantum reservoir** and a classical readout.

By the end, you will be able to:

1. explain the `feature → reservoir → probabilities → readout → next token` pipeline;
2. build a MerLin `CircuitBuilder` and `QuantumLayer`;
3. distinguish the reservoir's **fixed** parameters from the **trained** readout;
4. train with *teacher forcing*, then generate in a closed loop;
5. measure the temperature-controlled trade-off between **originality and broken transitions**.

> **Important — what is reproduced.** The original paper uses a qubit-based reservoir. Here, we preserve its algorithmic logic, data, metrics, leaking rate, and temperature mechanism, while translating the reservoir into a MerLin photonic circuit. This is therefore an **educational reproduction + photonic translation**, not an exact numerical replication.

There are only **seven short exercises**. The goal is to understand MerLin objects and the scientific logic, not to reimplement a quantum simulator.

In [ ]:
%pip install -q "merlinquantum>=0.4,<0.5" matplotlib pandas pillow

## 0. Set up the environment

This notebook targets **MerLin 0.4.x**. In a fresh environment, uncomment the first line in the next cell and restart the kernel if Jupyter asks you to.

In [ ]:
# Install once in a fresh environment:
# %pip install -q "merlinquantum>=0.4,<0.5" matplotlib pandas pillow

from __future__ import annotations

import base64
import io
import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

try:
    import torch
    import merlin as ML
except ImportError as exc:
    raise ImportError(
        "MerLin is not installed. Run the %pip install line above, "
        "then restart the kernel."
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("MerLin :", getattr(ML, "__version__", "version not exposed"))
print("PyTorch :", torch.__version__)
print("Device  : CPU (simulation locale)")


## 1. The paper in one loop

At time $t$:

1. the current token $x_t$ and memory $h_t$ are encoded into the reservoir;
2. measurement produces a probability vector $p_t$;
3. a classical readout produces the logits $y_t$;
4. the next token is sampled with temperature $T$;
5. memory is updated using a leaking rate of $\varepsilon=0.3$.

$$x_{t+1}\sim\mathrm{Categorical}\left(\mathrm{softmax}(y_t/T)\right)$$

$$h_{t+1}=(1-\varepsilon)h_t+\varepsilon p_t$$

### What learns — and what does not

| Component | Role | Trained? |
|---|---|---:|
| Encoding | Converts the token + memory into circuit parameters | No, randomly sampled and then fixed |
| Photonic reservoir | Projects the input to a rich distribution | No, fixed |
| Linear readout | Predicts the next token | **Yes** |
| Temperature $T$ | Controls diversity during generation | No, selected after training |

💬 **Before you continue:** what might happen if $T$ is very small? What if it is very large?

## 2. The data: a level becomes a sequence

Mario level 1-2 is split into columns that are 16 pixels wide. Each visually unique column becomes an integer feature. The level used in the paper contains **157 columns** and **32 unique features**.

The next cell contains the image and sequence directly in the notebook, so no external files are needed.

In [ ]:
# 🧰 Embedded data: Mario level 1-2 modified by the authors.

ORIGINAL_SEQUENCE = [
    0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 11, 11, 11, 11, 7, 7, 8, 6, 9,
    6, 10, 6, 12, 6, 12, 6, 10, 6, 17, 6, 10, 6, 9, 6, 6, 6, 6, 6,
    22, 15, 22, 20, 20, 22, 15, 31, 6, 6, 6, 6, 6, 23, 23, 26, 26, 6,
    6, 27, 15, 27, 27, 30, 30, 6, 6, 6, 30, 27, 29, 6, 6, 23, 24, 6,
    6, 28, 25, 28, 25, 5, 5, 5, 6, 21, 21, 21, 21, 21, 21, 6, 6, 6,
    6, 6, 6, 6, 6, 6, 7, 7, 6, 7, 13, 14, 6, 6, 6, 6, 18, 19, 6, 6,
    7, 6, 13, 14, 6, 6, 6, 5, 5, 10, 10, 5, 5, 6, 6, 6, 6, 6, 7, 6,
    8, 9, 10, 12, 16, 4, 4, 4, 3, 3, 3, 3, 3, 2, 0, 0, 4, 4, 4, 1,
    1, 1, 1, 1, 1,
]

_LEVEL_PNG_B64 = "..."
ORIGINAL_LEVEL_IMAGE = Image.open(
    io.BytesIO(base64.b64decode(_LEVEL_PNG_B64))
).convert("RGB")


def build_tile_atlas(image, sequence, tile_width=16):
    """Reconstruct feature_id -> 16 px column from the original level."""
    if image.width != tile_width * len(sequence):
        raise ValueError("Image width does not match the sequence length.")
    atlas = {}
    arrays = {}
    for index, feature in enumerate(sequence):
        tile = image.crop((index * tile_width, 0, (index + 1) * tile_width, image.height))
        tile_array = np.asarray(tile)
        if feature not in atlas:
            atlas[feature] = tile.copy()
            arrays[feature] = tile_array
        elif not np.array_equal(arrays[feature], tile_array):
            raise ValueError(f"Feature {feature} is not visually consistent.")
    return atlas


TILE_ATLAS = build_tile_atlas(ORIGINAL_LEVEL_IMAGE, ORIGINAL_SEQUENCE)


def render_level(sequence, atlas=TILE_ATLAS):
    """Assemble the tiles corresponding to a feature sequence."""
    tile_width = next(iter(atlas.values())).width
    height = next(iter(atlas.values())).height
    canvas = Image.new("RGB", (tile_width * len(sequence), height))
    for index, feature in enumerate(sequence):
        canvas.paste(atlas[int(feature)], (index * tile_width, 0))
    return canvas


display(ORIGINAL_LEVEL_IMAGE)


### ✍️ Exercise 1 — Inspect the dataset (3 min)

Complete the two expressions. Do not enter `157` and `32` directly: calculate them from the sequence.

In [ ]:
sequence_length = ...          # TODO: number of tokens
vocabulary_size = ...          # TODO: number of tokens distincts
 
if Ellipsis in (sequence_length, vocabulary_size):
    print("To complete: use len(...) and set(...).")
else:
    assert sequence_length == 157
    assert vocabulary_size == 32
    print(f"Sequence: {sequence_length} tokens | Vocabulary: {vocabulary_size} features")


In [ ]:
# ✅ Checkpoint 1 — solution
sequence_length = len(ORIGINAL_SEQUENCE)
vocabulary_size = len(set(ORIGINAL_SEQUENCE))

assert sequence_length == 157 and vocabulary_size == 32
print(f"Sequence: {sequence_length} tokens | Vocabulary: {vocabulary_size} features")


### ✍️ Exercise 2 — Build the next-token task (4 min)

With *teacher forcing*, the input is the true feature at time $t$, and the target is the true feature at time $t+1$.

Complete only `targets`.

In [ ]:
inputs = np.asarray(ORIGINAL_SEQUENCE[:-1], dtype=np.int64)
targets = ...  # TODO: same sequence, shifted one step to the left

if targets is Ellipsis:
    print("Hint: the first target should be ORIGINAL_SEQUENCE[1].")
else:
    assert len(inputs) == len(targets) == 156
    print(list(zip(inputs[:8], targets[:8])))


In [ ]:
# ✅ Checkpoint 2 — solution
inputs = np.asarray(ORIGINAL_SEQUENCE[:-1], dtype=np.int64)
targets = np.asarray(ORIGINAL_SEQUENCE[1:], dtype=np.int64)

assert len(inputs) == len(targets) == 156
print("First pairs (x_t, x_{t+1}):", list(zip(inputs[:8], targets[:8])))


## 3. Build the photonic reservoir in MerLin — 20 min

We translate the paper's 6-qubit reservoir into a compact photonic reservoir:

| Choice | Value | Consequence |
|---|---:|---|
| Optical modes $m$ | 6 | six inputs encoded as phases |
| Photons $n$ | 3 | multiphoton interference |
| Computation space | `UNBUNCHED` | at most one photon per mode |
| Measured dimension | $\binom{6}{3}=20$ | the readout receives 20 probabilities |
| Quantum parameters | fixed | only the readout learns |

The circuit follows this pattern:

`random interferometer → angle encoding → interferometer → interferometer → probabilities`

`CircuitBuilder` describes the circuit. `QuantumLayer` turns it into a PyTorch module that works with the rest of the pipeline.

### ✍️ Exercise 3 — Add MerLin encoding (5 min)

In the function below, replace `...` with **one call** to `builder.add_angle_encoding(...)`:

- target all six modes;
- name the input `qrc_input`;
- use a scale of $\pi$.

This function is a draft: the checkpoint cell will build the actual reservoir afterward so nobody gets stuck.

In [ ]:
N_MODES = 6
N_PHOTONS = 3


def build_student_circuit():
    builder = ML.CircuitBuilder(n_modes=N_MODES)
    builder.add_entangling_layer(trainable=True, name="pre")

    ...  # TODO : builder.add_angle_encoding(...)

    builder.add_entangling_layer(trainable=True, name="post_1")
    builder.add_entangling_layer(trainable=True, name="post_2")
    return builder


In [ ]:
# ✅ Checkpoint 3 — build the MerLin reservoir used below


def build_merlin_reservoir(n_modes=6, n_photons=3):
    builder = ML.CircuitBuilder(n_modes=n_modes)
    builder.add_entangling_layer(trainable=True, name="pre")
    builder.add_angle_encoding(
        modes=list(range(n_modes)),
        name="qrc_input",
        scale=np.pi,
    )
    builder.add_entangling_layer(trainable=True, name="post_1")
    builder.add_entangling_layer(trainable=True, name="post_2")

    layer = ML.QuantumLayer(
        input_size=n_modes,
        builder=builder,
        n_photons=n_photons,
        measurement_strategy=ML.MeasurementStrategy.probs(
            computation_space=ML.ComputationSpace.UNBUNCHED
        ),
        dtype=torch.float64,
    )

    # Reservoir computing: the circuit is random but is not trained.
    for parameter in layer.parameters():
        parameter.requires_grad_(False)
    layer.eval()
    return layer


reservoir = build_merlin_reservoir(N_MODES, N_PHOTONS)
RESERVOIR_DIM = reservoir.output_size

expected_dim = math.comb(N_MODES, N_PHOTONS)
assert RESERVOIR_DIM == expected_dim == 20

with torch.no_grad():
    probe = reservoir(torch.zeros((1, N_MODES), dtype=torch.float64))

print(reservoir)
print("Output shape:", tuple(probe.shape))
print("Sum of probabilities:", float(probe.sum()))
print("Trainable quantum parameters:", sum(p.requires_grad for p in reservoir.parameters()))


### Inspect the object instead of treating it as a black box

Three checks should pass:

1. the output has 20 components;
2. they form a probability distribution;
3. no reservoir parameter has `requires_grad=True`.

💬 **Question:** why can a fixed random projection still be useful?  
*Hint: the reservoir transforms the input into a space where a linear separator may be sufficient.*

## 4. Inject the token and memory — 10 min

The reservoir expects six real values. We therefore prepare these once:

- a random angle lookup table, `feature → 6 values`;
- a random projection, `20-dimensional memory → 6 values`.

Both objects are fixed. They play the role of the non-trainable encoding described in the reproduction.

In [ ]:
# 🧰 Provided encoding: token + memory -> 6 MerLin inputs in [-1, 1]


class FixedQRCEncoder:
    def __init__(self, num_features, n_modes, reservoir_dim, seed=42):
        rng = np.random.default_rng(seed)
        self.feature_book = rng.uniform(-1.0, 1.0, size=(num_features, n_modes))
        self.hidden_projection = rng.normal(0.0, 1.0, size=(reservoir_dim, n_modes))
        self.reservoir_dim = reservoir_dim

    def __call__(self, token, hidden):
        token_signal = self.feature_book[int(token)]
        memory_signal = np.sqrt(self.reservoir_dim) * (hidden @ self.hidden_projection)
        return np.tanh(token_signal + memory_signal)


encoder = FixedQRCEncoder(
    num_features=vocabulary_size,
    n_modes=N_MODES,
    reservoir_dim=RESERVOIR_DIM,
    seed=SEED,
)

initial_hidden = np.full(RESERVOIR_DIM, 1.0 / RESERVOIR_DIM)
encoded_example = encoder(ORIGINAL_SEQUENCE[0], initial_hidden)

print("MerLin input:", np.round(encoded_example, 3))
print("Shape :", encoded_example.shape)


### ✍️ Exercise 4 — Update the memory (4 min)

Implement the equation from the paper:

$$h_{t+1}=(1-\varepsilon)h_t+\varepsilon p_t$$

In [ ]:
def update_hidden_student(hidden, probabilities, epsilon=0.3):
    return ...  # TODO: a single expression


In [ ]:
# ✅ Checkpoint 4 — solution
def update_hidden(hidden, probabilities, epsilon=0.3):
    updated = (1.0 - epsilon) * hidden + epsilon * probabilities
    return updated / updated.sum()


test_probabilities = np.zeros(RESERVOIR_DIM)
test_probabilities[3] = 1.0
test_hidden = update_hidden(initial_hidden, test_probabilities)
assert np.isclose(test_hidden.sum(), 1.0)
print("Memory remains a distribution:", np.isclose(test_hidden.sum(), 1.0))


## 5. Teacher forcing: turn the level into quantum features — 15 min

During training, the model always sees the true current token. The reservoir computes $p_t$, memory is updated, and then $p_t$ becomes the readout input.

The next cell makes 156 sequential MerLin calls. On a CPU, this may take a few moments.

In [ ]:
# 🧰 Provided quantum feature extraction


def reservoir_probabilities(layer, encoded_input):
    tensor = torch.as_tensor(encoded_input, dtype=torch.float64).unsqueeze(0)
    with torch.no_grad():
        probabilities = layer(tensor).squeeze(0).detach().cpu().numpy()
    probabilities = np.clip(probabilities.astype(float), 0.0, None)
    return probabilities / probabilities.sum()


def teacher_forced_features(layer, qrc_encoder, sequence, epsilon=0.3):
    hidden = np.full(layer.output_size, 1.0 / layer.output_size)
    features = []

    for step, token in enumerate(sequence[:-1]):
        encoded_input = qrc_encoder(token, hidden)
        probabilities = reservoir_probabilities(layer, encoded_input)
        features.append(probabilities)
        hidden = update_hidden(hidden, probabilities, epsilon)

        if (step + 1) % 40 == 0:
            print(f"  {step + 1:>3}/{len(sequence) - 1} steps computed")

    return np.asarray(features)


quantum_features = teacher_forced_features(
    reservoir,
    encoder,
    ORIGINAL_SEQUENCE,
    epsilon=0.3,
)

assert quantum_features.shape == (156, RESERVOIR_DIM)
print("Quantum feature matrix:", quantum_features.shape)


### Train only the readout

The readout is a linear `20 → 32` layer. It is the only part optimized by Adam. The loss is cross-entropy on the next token.

In [ ]:
# 🧰 Provided classical readout training

X = torch.as_tensor(quantum_features, dtype=torch.float64)
y = torch.as_tensor(targets, dtype=torch.long)

torch.manual_seed(SEED)
readout = torch.nn.Linear(RESERVOIR_DIM, vocabulary_size, dtype=torch.float64)
optimizer = torch.optim.Adam(readout.parameters(), lr=0.05)
loss_function = torch.nn.CrossEntropyLoss()

loss_history = []
for epoch in range(200):
    optimizer.zero_grad()
    logits = readout(X)
    loss = loss_function(logits, y)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.item()))

readout.eval()

plt.figure(figsize=(6.5, 3.2))
plt.plot(loss_history, color="#5B43D6")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.title("Only the classical readout is trained")
plt.grid(alpha=0.2)
plt.show()

print(f"Initial loss: {loss_history[0]:.3f}")
print(f"Final loss: {loss_history[-1]:.3f}")
print("Trained parameters:", sum(p.numel() for p in readout.parameters()))


## 6. Generate in a closed loop — 20 min

During training, the next input is known. During generation, the next input is **the previous prediction**. An error can therefore propagate; this is exactly what the gameplay metrics are meant to reveal.

### ✍️ Exercise 5 — Add temperature (3 min)

Complete the logit scaling. A low temperature should make the distribution more peaked.

In [ ]:
def softmax_temperature_student(logits, temperature=1.0):
    assert temperature > 0
    scaled = ...  # TODO : que faut-il faire aux logits ?
    scaled = scaled - np.max(scaled)
    exponentials = np.exp(scaled)
    return exponentials / exponentials.sum()


In [ ]:
# ✅ Checkpoint 5 — solution
def softmax_temperature(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be strictly positive")
    scaled = np.asarray(logits, dtype=float) / float(temperature)
    scaled = scaled - np.max(scaled)
    exponentials = np.exp(scaled)
    return exponentials / exponentials.sum()


demo_logits = np.array([3.0, 1.0, 0.0, -1.0])
for temperature in [0.1, 1.0, 10.0]:
    print(f"T={temperature:<4}", np.round(softmax_temperature(demo_logits, temperature), 3))


### ✍️ Exercise 6 — Sample the next token (3 min)

In the draft, complete the call to `rng.choice`. The executable solution follows immediately.

In [ ]:
def sample_next_token_student(logits, temperature, rng):
    probabilities = softmax_temperature(logits, temperature)
    next_token = ...  # TODO : rng.choice(vocabulary_size, p=probabilities)
    return int(next_token)


In [ ]:
# ✅ Checkpoint 6 — complete generator
def sample_next_token(logits, temperature, rng):
    probabilities = softmax_temperature(logits, temperature)
    return int(rng.choice(len(probabilities), p=probabilities))


def generate_level(
    layer,
    qrc_encoder,
    classical_readout,
    length,
    temperature,
    seed,
    epsilon=0.3,
    first_token=0,
):
    rng = np.random.default_rng(seed)
    hidden = np.full(layer.output_size, 1.0 / layer.output_size)
    generated = [int(first_token)]

    for _ in range(length - 1):
        current_token = generated[-1]
        encoded_input = qrc_encoder(current_token, hidden)
        probabilities = reservoir_probabilities(layer, encoded_input)
        hidden = update_hidden(hidden, probabilities, epsilon)

        with torch.no_grad():
            logits = classical_readout(
                torch.as_tensor(probabilities, dtype=torch.float64)
            ).detach().cpu().numpy()

        generated.append(sample_next_token(logits, temperature, rng))

    return generated


### Run the mini temperature sweep

Workshop mode generates three levels per temperature. Set `FULL_REPRODUCTION = True` after the class to get more stable estimates with 20 levels per temperature.

💬 **Before running it:** rank `T = 0.7`, `1.0`, and `2.0` from the most conservative expected level to the most creative.

In [ ]:
FULL_REPRODUCTION = False

TEMPERATURES = [0.7, 1.0, 2.0]
N_GENERATIONS = 20 if FULL_REPRODUCTION else 3

qrc_generations = {}
for temperature in TEMPERATURES:
    qrc_generations[temperature] = [
        generate_level(
            reservoir,
            encoder,
            readout,
            length=len(ORIGINAL_SEQUENCE),
            temperature=temperature,
            seed=SEED + sample_id + int(100 * temperature),
            first_token=ORIGINAL_SEQUENCE[0],
        )
        for sample_id in range(N_GENERATIONS)
    ]
    print(f"T={temperature}: {N_GENERATIONS} level(s) generated")


In [ ]:
# Visual comparison: one level per temperature
for temperature in TEMPERATURES:
    print(f"T = {temperature}")
    display(render_level(qrc_generations[temperature][0]))


## 7. Always compare against baselines — 10 min

- **Uncorrelated** preserves only the global frequency of each feature.
- **Markov** preserves the distribution of the next token conditioned on the current token.

If QRC does not outperform them on the trade-off being studied, this experiment does not justify the quantum complexity.

In [ ]:
# 🧰 Provided baselines
def fit_feature_prior(sequence):
    counts = Counter(sequence)
    values = np.asarray(sorted(counts), dtype=int)
    probabilities = np.asarray([counts[value] for value in values], dtype=float)
    return values, probabilities / probabilities.sum()


def fit_markov(sequence):
    transitions = defaultdict(Counter)
    for index, current in enumerate(sequence):
        next_token = sequence[(index + 1) % len(sequence)]
        transitions[int(current)][int(next_token)] += 1
    return transitions


def generate_uncorrelated(sequence, length, rng):
    values, probabilities = fit_feature_prior(sequence)
    return rng.choice(values, size=length, p=probabilities).astype(int).tolist()


def generate_markov(sequence, length, rng):
    transitions = fit_markov(sequence)
    output = [int(rng.choice(sequence))]
    for _ in range(length - 1):
        counter = transitions[output[-1]]
        values = np.asarray(sorted(counter), dtype=int)
        probabilities = np.asarray([counter[value] for value in values], dtype=float)
        probabilities /= probabilities.sum()
        output.append(int(rng.choice(values, p=probabilities)))
    return output


baseline_count = 100
baseline_rng = np.random.default_rng(SEED)

uncorrelated_generations = [
    generate_uncorrelated(ORIGINAL_SEQUENCE, len(ORIGINAL_SEQUENCE), baseline_rng)
    for _ in range(baseline_count)
]
markov_generations = [
    generate_markov(ORIGINAL_SEQUENCE, len(ORIGINAL_SEQUENCE), baseline_rng)
    for _ in range(baseline_count)
]

print("Baselines generated:", len(markov_generations), "Markov et", len(uncorrelated_generations), "uncorrelated")


## 8. Measure originality and gameplay errors — 15 min

### Originality rate

For a sequence length $L$, count the generated subsequences that never appear in the original level. A higher value means the level combines more novel patterns.

### Broken-transition rate

We use rule group “2” from the authors' notebook: correctly paired pipe sides and unobstructed passages. Lower is better.

A good generation is therefore in the **upper-left** of a chart with `broken rate` on the x-axis and `originality` on the y-axis.

In [ ]:
# 🧰 Metrics faithful to the source notebook — provided
def count_ngrams(sequence, length):
    return Counter(
        tuple(sequence[index : index + length])
        for index in range(len(sequence) - length + 1)
    )


def originality_curve(generated_sequences, original, max_length=20):
    scores = {}
    for length in range(2, max_length + 1):
        original_ngrams = count_ngrams(original, length)
        novel_occurrences = 0
        total_occurrences = 0

        for generated in generated_sequences:
            generated_ngrams = count_ngrams(generated, length)
            for ngram, count in generated_ngrams.items():
                total_occurrences += count
                if ngram not in original_ngrams:
                    novel_occurrences += count

        scores[length] = novel_occurrences / total_occurrences
    return scores


def mario_rule_2(index, sequence):
    "Return True/False if a rule applies, otherwise None."
    if index <= 0 or index >= len(sequence) - 1:
        return None

    feature = int(sequence[index])
    previous_feature = int(sequence[index - 1])
    next_feature = int(sequence[index + 1])

    if feature == 14:  # right side of the small pipe
        return previous_feature == 13
    if feature == 13:  # left side of the small pipe
        return next_feature == 14
    if feature == 19:  # right side of the large pipe
        return previous_feature == 18
    if feature == 18:  # left side of the large pipe
        return next_feature == 19
    if feature == 30:  # unobstructed passage
        blockers = {1, 10, 12, 16, 18, 19}
        return previous_feature not in blockers and next_feature not in blockers
    return None


def broken_transition_rate(sequence):
    outcomes = [
        mario_rule_2(index, sequence)
        for index in range(len(sequence))
    ]
    applicable = [outcome for outcome in outcomes if outcome is not None]
    if not applicable:
        return np.nan
    return 1.0 - float(np.mean(applicable))


def mean_broken_rate(sequences):
    return float(np.nanmean([broken_transition_rate(sequence) for sequence in sequences]))


In [ ]:
# Originality curves
curves = {
    "Markov": originality_curve(markov_generations, ORIGINAL_SEQUENCE),
    "Uncorrelated": originality_curve(uncorrelated_generations, ORIGINAL_SEQUENCE),
}
for temperature in TEMPERATURES:
    curves[f"MerLin T={temperature}"] = originality_curve(
        qrc_generations[temperature], ORIGINAL_SEQUENCE
    )

plt.figure(figsize=(8.5, 5.0))
for label, curve in curves.items():
    style = "--" if label in {"Markov", "Uncorrelated"} else "-o"
    plt.plot(list(curve), list(curve.values()), style, markersize=3, label=label)

plt.xlabel("Subsequence length L")
plt.ylabel("Originality rate")
plt.ylim(-0.03, 1.03)
plt.grid(alpha=0.2)
plt.legend()
plt.title("Novelty grows with window length — but gameplay may break")
plt.show()


In [ ]:
# Summary table and Pareto frontier
rows = []

for label, sequences in [
    ("Markov", markov_generations),
    ("Uncorrelated", uncorrelated_generations),
]:
    curve = originality_curve(sequences, ORIGINAL_SEQUENCE)
    rows.append({
        "generator": label,
        "temperature": np.nan,
        "originality_L2": curve[2],
        "originality_L10": curve[10],
        "broken_rate": mean_broken_rate(sequences),
    })

for temperature, sequences in qrc_generations.items():
    curve = originality_curve(sequences, ORIGINAL_SEQUENCE)
    rows.append({
        "generator": "MerLin photonic QRC",
        "temperature": temperature,
        "originality_L2": curve[2],
        "originality_L10": curve[10],
        "broken_rate": mean_broken_rate(sequences),
    })

results = pd.DataFrame(rows)
display(results.round(3))

plt.figure(figsize=(7.0, 4.8))
for _, row in results.iterrows():
    marker = "o" if row["generator"] == "MerLin photonic QRC" else "s"
    plt.scatter(row["broken_rate"], row["originality_L2"], s=85, marker=marker)
    label = row["generator"]
    if not np.isnan(row["temperature"]):
        label += f" T={row['temperature']}"
    plt.annotate(label, (row["broken_rate"], row["originality_L2"]), xytext=(5, 5), textcoords="offset points")

plt.xlabel("Broken-transition rate ↓")
plt.ylabel("Originality at L=2 ↑")
plt.title("Quality is a trade-off, not a single score")
plt.grid(alpha=0.2)
plt.show()


### Published reference values

These values are **reference points**, not results to expect cell by cell: the reservoir is random, workshop mode uses few generations, and the photonic translation has a different dimension from the qubit-based reservoir.

| System | Orig. L=2, T=1 | Orig. L=10, T=1 | Broken rate, T=1 |
|---|---:|---:|---:|
| Paper sequences | 0.063 | 0.695 | 0.003 |
| Markov | 0.007 | 0.992 | 0.000 |
| Uncorrelated | 0.627 | 1.000 | 0.731 |
| Reference photonic translation | 0.274 | 1.000 | 0.578 |

The robust conclusion is not “MerLin must produce exactly 0.274,” but rather: **temperature moves the model along a trade-off between novelty and coherence**.

### ✍️ Exercise 7 — Defend a choice (5 min)

Using your `results` table, choose a temperature for two different goals.

In [ ]:
# There is no single right answer: justify your choice using two metrics.
temperature_for_safe_generation = ...
temperature_for_creative_generation = ...

evidence = (
    "Safe generation: ...\n"
    "Creative generation: ..."
)

print(evidence)


## 9. What you actually did — 5 min

### Faithfully reproduced

- Mario level 1-2 encoded as 157 tokens / 32 features;
- autoregressive next-token prediction;
- a fixed reservoir + trained classical readout;
- memory with $\varepsilon=0.3$;
- temperature applied after training;
- Markov and uncorrelated baselines;
- the originality rate and gameplay rule group “2.”

### Adapted for MerLin

- 6-qubit reservoir, $2^6=64$ → 6-mode / 3-photon reservoir, dimension 20;
- `Ry` rotations → photonic phase angle encoding;
- random X/H/CNOT circuit → fixed random interferometers;
- local analytical simulation, without shots or a QPU.

### Not demonstrated by this mini-run

- a quantum advantage;
- an exact numerical replication of the paper's figures;
- statistical robustness across many seeds;
- execution on hardware;
- the noise sweep and Roblox experiment.

> The scientifically accurate wording is: **“We reproduce the mechanism and qualitative trend in a MerLin photonic translation.”**

## 10. After the workshop: move from the notebook to the full benchmark

For a more robust analysis:

1. set `FULL_REPRODUCTION = True`;
2. repeat the experiment with multiple circuit and sampling seeds;
3. report the mean ± standard deviation;
4. compare with the gate-based reproduction in the repository;
5. test several `(modes, photons)` pairs with comparable output dimensions;
6. preserve the configuration, package versions, and artifacts for every run.

The full repository remains the right interface for traceable experiments: it centralizes configurations, tests, runners, and results. This notebook is the **single educational layer** that exposes the scientific thread without asking students to rewrite those components.

## Instructor guide — keep this at the bottom of the notebook

- Ask participants to state their predictions **before** running the plots.
- Spend no more than 5 minutes on any fill-in cell: run the checkpoint and move on.
- Section 3 is the MerLin core: emphasize `CircuitBuilder`, `QuantumLayer`, `MeasurementStrategy`, and freezing the parameters.
- If computation is slow, generate one level per temperature by setting `N_GENERATIONS = 1`.
- If you need to shorten the workshop to 90 minutes, skip manually building the baselines and provide their results directly.
- Reserve 5 minutes to distinguish **reproduction**, **adaptation**, and **evidence of quantum advantage**.

### Closing questions

1. Why can the readout learn while the reservoir remains fixed?
2. Where is the model's memory stored?
3. Why does changing the temperature not require retraining?
4. Is high originality enough to conclude that a level is good?
5. What additional experiment would make the conclusion more convincing?

## References

- Ferreira, Fromholz, Shaji & Wootton, *Level Generation with Quantum Reservoir Computing*, arXiv:2505.13287.
- [MerLin reproduction — qrc_level_generation](https://github.com/merlinquantum/reproduced_papers/tree/main/papers/qrc_level_generation)
- [MerLin 0.4 documentation](https://merlinquantum.ai/0.4/)
- [QuantumLayer Essentials](https://merlinquantum.ai/0.4/user_guide/layer.html)
- [CircuitBuilder / first quantum layer](https://merlinquantum.ai/0.4/quickstart/first_quantum_layer.html)
- [Computation spaces](https://merlinquantum.ai/0.4/user_guide/computation_space.html)

The Mario data and metric rules included here come from the reproduction materials provided with the paper. The photonic circuit follows the MerLin translation described in the reproduction's Executive Summary.